# CharTokenizer — 라이브러리로 사용하기

이 노트북은 `char_tokenizer.py`를 **외부 라이브러리처럼 import**해서  
실제 텍스트 전처리 파이프라인에서 어떻게 쓰는지 보여줍니다.

개념 학습은 `01_char_tokenizer.ipynb`에서 끝났습니다.  
여기서는 **'쓰는 것'** 에 집중합니다.

---

## 이 노트북에서 다루는 것

1. 라이브러리 import 패턴
2. 실제 텍스트 데이터로 학습
3. 인코딩 결과를 PyTorch Tensor로 변환
4. 배치 처리 파이프라인
5. 어휘집 저장 → 새 환경에서 불러오기
6. 실전 시나리오: 간단한 텍스트 분류 전처리

## Step 0. 라이브러리 import

`char_tokenizer.py`가 같은 폴더에 있으면 아래처럼 바로 import됩니다.  
실무에서는 패키지로 설치하지만, 지금은 파일 경로를 직접 추가하는 방식을 사용합니다.

In [ ]:
import sys
import os

# 이 노트북이 있는 폴더를 Python 경로에 추가합니다.
# 그러면 같은 폴더의 .py 파일을 import할 수 있습니다.
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

# char_tokenizer.py에서 CharTokenizer 클래스를 가져옵니다.
from char_tokenizer import CharTokenizer

print("CharTokenizer import 성공!")
print(f"사용 가능한 메서드: {[m for m in dir(CharTokenizer) if not m.startswith('_')]}")

## Step 1. 실제 데이터로 학습 — 한국어 위키피디아

짧은 예문 대신 **실제로 다운로드한 한국어 위키피디아 덤프**로 바로 학습합니다.

- 파일: `data/kowiki/kowiki-latest-pages-articles.xml.bz2` (1.2GB)
- `SAMPLE_MB` 값을 바꿔 학습 데이터 양을 조절할 수 있습니다
- 이후 Step들은 이 `tokenizer` 객체를 계속 사용합니다

In [ ]:
import bz2, re, os, pathlib

# CWD에서 위로 올라가며 data/kowiki 폴더를 자동 탐색
def find_kowiki():
    target = pathlib.Path("data") / "kowiki" / "kowiki-latest-pages-articles.xml.bz2"
    for parent in [pathlib.Path.cwd()] + list(pathlib.Path.cwd().parents):
        candidate = parent / target
        if candidate.exists():
            return candidate  # pathlib.Path 객체로 반환
    return None

kowiki_path = find_kowiki()
if kowiki_path is None:
    raise FileNotFoundError("kowiki 파일을 찾을 수 없습니다.")

# kowiki_path = .../GPT-2/data/kowiki/kowiki-latest-pages-articles.xml.bz2
# .parent x3 → GPT-2 프로젝트 루트
project_root = kowiki_path.parent.parent.parent
print(f"프로젝트 루트: {project_root}")
print(f"파일 경로:     {kowiki_path}")

# XML/위키 문법 정제 함수
def clean_wiki_text(text):
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\[\[([^|\]]+\|)?([^\]]+)\]\]", r"\2", text)
    text = re.sub(r"\{\{[^}]*\}\}", " ", text)
    text = re.sub(r"[=\-\*#\|]{2,}", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# 샘플 크기 설정
# 2  → 2MB  (빠름, 약 5초)
# 10 → 10MB (보통, 약 20초)
# None → 전체 파일 (수십 분 소요)
SAMPLE_MB = None
SAMPLE_BYTES = SAMPLE_MB * 1024 * 1024 if SAMPLE_MB is not None else None

label = f"{SAMPLE_MB}MB" if SAMPLE_MB else "전체"
print(f"\n{label} 읽는 중...")

with bz2.open(kowiki_path, "rt", encoding="utf-8") as f:
    raw = f.read(SAMPLE_BYTES) if SAMPLE_BYTES else f.read()

train_corpus = clean_wiki_text(raw)
print(f"정제 완료: {len(train_corpus):,} 글자")

# CharTokenizer 학습
tokenizer = CharTokenizer()
tokenizer.train(train_corpus)

# 어휘집 분석
vocab = tokenizer.get_vocab()
hangul = [c for c in vocab if "가" <= c <= "힣"]
latin  = [c for c in vocab if c.isascii() and c.isalpha()]
print(f"\n한글 글자: {len(hangul)}자")
print(f"영문 글자: {len(latin)}자")
print(f"한글 예시: {sorted(hangul)[:15]}")

# 학습 직후 JSON으로 저장
save_path = project_root / "tokenizer" / "src" / "stage1_char" / "kowiki_char_tokenizer.json"
tokenizer.save(str(save_path))
print(f"\n저장 완료: {save_path}")
print(f"파일 크기: {save_path.stat().st_size / 1024:.1f} KB")

## Step 2. 기본 인코딩 / 디코딩

In [ ]:
# 한국어 / 영어 혼합 문장으로 인코딩 테스트
test_sentences = [
    "대한민국",
    "인공지능 언어 모델",
    "위키피디아는 자유 백과사전입니다",
    "AI is amazing",
]

for sentence in test_sentences:
    try:
        encoded = tokenizer.encode(sentence, add_special_tokens=True)
        decoded = tokenizer.decode(encoded, skip_special_tokens=True)
        match = "✓" if decoded == sentence else "✗"
        print(f"{match} '{sentence}'")
        print(f"   토큰 수: {len(encoded)}개  |  인코딩: {encoded[:8]}{'...' if len(encoded)>8 else ''}")
        print(f"   디코딩: '{decoded}'")
    except Exception as e:
        print(f"✗ '{sentence}' → 오류: {e}")
    print()

## Step 3. 인코딩 결과를 PyTorch Tensor로 변환

실제 모델에 넣으려면 **숫자 ID 목록 → PyTorch Tensor** 변환이 필요합니다.  
토크나이저 출력을 모델 입력으로 연결하는 과정입니다.

In [ ]:
import torch

sentence = "The quick fox"

# 1단계: 텍스트 → 숫자 ID 목록
ids = tokenizer.encode(sentence, add_special_tokens=True)
print(f"텍스트:  '{sentence}'")
print(f"ID 목록: {ids}")

# 2단계: 숫자 ID 목록 → PyTorch Tensor
# unsqueeze(0)은 배치 차원을 추가합니다. 모델은 배치 단위로 입력을 받습니다.
# shape: (1, 시퀀스_길이) → 배치 1개, 토큰 n개
tensor = torch.tensor(ids).unsqueeze(0)
print(f"\nTensor: {tensor}")
print(f"shape:  {tensor.shape}  ← (배치 크기, 시퀀스 길이)")
print(f"dtype:  {tensor.dtype}")

## Step 4. 배치 처리 파이프라인

실제 학습에서는 여러 문장을 한 번에 처리합니다.  
길이가 다른 문장들을 PAD로 맞추고 Tensor로 만드는 과정입니다.

In [ ]:
import torch

# 길이가 다른 여러 문장
batch_sentences = [
    "The quick brown fox",
    "To be",
    "All that glitters is not gold.",
    "gold",
]

# 배치 인코딩 (PAD로 길이 자동 통일)
batch_ids = tokenizer.encode_batch(
    batch_sentences,
    add_special_tokens=True,
    pad_to_max_length=True
)

print("배치 인코딩 결과 (PAD=0):")
for i, (sent, ids) in enumerate(zip(batch_sentences, batch_ids)):
    print(f"  [{i}] '{sent[:20]:<20}' → {ids}")

# 2D Tensor로 변환 → 모델 입력 형태
batch_tensor = torch.tensor(batch_ids)
print(f"\nbatch_tensor.shape: {batch_tensor.shape}  ← (배치 크기, 최대 시퀀스 길이)")

# Attention Mask: PAD 위치는 0, 나머지는 1
# 모델이 PAD를 무시하도록 알려주는 마스크입니다
PAD_ID = 0
attention_mask = (batch_tensor != PAD_ID).long()
print(f"\nAttention Mask:")
print(attention_mask)

## Step 5. 어휘집 저장 → 다른 환경에서 불러오기

**실무 시나리오**: 서버 A에서 학습한 토크나이저를 서버 B에서 사용하는 상황.  
JSON 파일 하나만 옮기면 됩니다.

In [ ]:
import json

# --- 서버 A: 학습 후 저장 ---
save_path = "tokenizer.json"
tokenizer.save(save_path)

# 저장된 JSON 구조 확인
with open(save_path, encoding="utf-8") as f:
    saved_data = json.load(f)

print("저장된 JSON 구조:")
print(f"  vocab_size: {saved_data['vocab_size']}")
print(f"  special_tokens: {saved_data['special_tokens']}")
print(f"  word_to_id (처음 8개): {dict(list(saved_data['char_to_id'].items())[:8])}")

# --- 서버 B: JSON 불러와서 바로 사용 ---
tokenizer_B = CharTokenizer()
tokenizer_B.load(save_path)

# A와 B의 인코딩 결과가 동일한지 확인
test = "The fox"
ids_A = tokenizer.encode(test)
ids_B = tokenizer_B.encode(test)

print(f"\n서버 A 인코딩: {ids_A}")
print(f"서버 B 인코딩: {ids_B}")
print(f"동일한가? {'✓ Yes' if ids_A == ids_B else '✗ No'}")

## Step 6. 실전 시나리오 — 감성 분류 전처리

간단한 감성 분류 데이터셋을 전처리하는 파이프라인입니다.  
토크나이저가 실제 ML 파이프라인에서 어떻게 사용되는지 보여줍니다.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# 가상의 감성 분류 데이터셋
# label: 1 = 긍정, 0 = 부정
raw_data = [
    ("The movie was great", 1),
    ("I love this", 1),
    ("Not good at all", 0),
    ("Totally boring", 0),
    ("Amazing story", 1),
    ("Waste of time", 0),
]

# 전체 텍스트로 토크나이저 학습
all_text = " ".join([text for text, _ in raw_data])
tok = CharTokenizer()
tok.train(all_text)
print(f"vocab_size: {tok.vocab_size}")


# PyTorch Dataset 클래스로 감싸기
class TextDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=30):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = []

        for text, label in data:
            ids = tokenizer.encode(text, add_special_tokens=True)
            # max_length로 자르거나 PAD로 채우기
            if len(ids) > max_length:
                ids = ids[:max_length]
            else:
                ids = ids + [0] * (max_length - len(ids))  # PAD = 0
            self.samples.append((ids, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ids, label = self.samples[idx]
        return torch.tensor(ids), torch.tensor(label)


dataset = TextDataset(raw_data, tok)
loader = DataLoader(dataset, batch_size=2, shuffle=False)

print("\n배치 단위로 데이터 순회:")
for batch_idx, (input_ids, labels) in enumerate(loader):
    print(f"\n  배치 {batch_idx}:")
    print(f"    input_ids.shape: {input_ids.shape}  ← (배치 크기, 시퀀스 길이)")
    print(f"    labels: {labels}")

## Step 7. vocab_size와 모델 임베딩의 관계

토크나이저의 `vocab_size`가 모델 설계에 어떻게 연결되는지 확인합니다.

In [ ]:
import torch.nn as nn

# vocab_size는 Embedding 레이어 크기를 결정합니다
vocab_size = tokenizer.vocab_size
embedding_dim = 16  # 실제 GPT-2는 768, 여기서는 학습용으로 작게 설정

# Embedding 레이어: 각 토큰 ID를 embedding_dim 차원 벡터로 변환
embedding = nn.Embedding(
    num_embeddings=vocab_size,   # vocab_size만큼의 행 (토큰 수)
    embedding_dim=embedding_dim  # 각 토큰을 16차원 벡터로 표현
)

print(f"vocab_size:     {vocab_size}")
print(f"embedding_dim:  {embedding_dim}")
print(f"임베딩 행렬 크기: {vocab_size} × {embedding_dim} = {vocab_size * embedding_dim:,}개 파라미터")
print(f"\nembedding weight shape: {embedding.weight.shape}")

# 실제 토큰을 임베딩 벡터로 변환해보기
test_ids = torch.tensor([[7, 3, 8, 8, 9]])  # 'hello'의 ID 예시
embedded = embedding(test_ids)
print(f"\n입력 IDs shape:    {test_ids.shape}  ← (배치 1, 토큰 5개)")
print(f"임베딩 출력 shape: {embedded.shape}  ← (배치 1, 토큰 5개, 벡터 16차원)")
print(f"\n첫 번째 토큰의 임베딩 벡터:")
print(embedded[0][0].detach())

## 정리

```
CharTokenizer 사용 패턴 요약

1. import
   from char_tokenizer import CharTokenizer

2. 학습
   tok = CharTokenizer()
   tok.train(corpus_text)

3. 인코딩 (텍스트 → ID)
   ids = tok.encode(text, add_special_tokens=True)

4. 배치 인코딩 (여러 문장 → 2D 목록)
   batch = tok.encode_batch(texts, pad_to_max_length=True)

5. Tensor 변환 (모델 입력)
   tensor = torch.tensor(batch)

6. 저장 / 불러오기
   tok.save('tokenizer.json')
   tok.load('tokenizer.json')

7. 디코딩 (ID → 텍스트)
   text = tok.decode(ids, skip_special_tokens=True)
```

다음 단계: `stage2_word/02_word_tokenizer.ipynb` 에서 단어 단위 토크나이저로 넘어갑니다.